# kline_de_pre — Kaggle 一次训练

在 Kaggle 上：**clone 代码 → 从 OKX 拉取能拿到的最多 BTC 1m 历史 → 从零训练两个模型**（CNN_Transformer + UD）。

> ⚠️ 先在右侧 **Settings → Accelerator 选 GPU**，并把 **Internet 打开**（要访问 OKX 与 GitHub）。

产出：`transformer_dct_v1.pth`（模型1）、`diffusion_delta_v1.pth`（模型2），会复制到 `/kaggle/working/` 供下载。

In [ ]:
# 1) 拉代码 + 依赖 + GPU 自检
import os
if not os.path.exists('kline_de_pre'):
    !git clone --depth 1 https://github.com/lin7c/kline_de_pre.git
%cd kline_de_pre
!pip -q install 'requests>=2.32' 'pandas>=2.2' 'numpy>=2.0' 'scipy>=1.13' 'scikit-learn>=1.5'
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(无 GPU，请在 Settings 里选 GPU)')

In [ ]:
# 2) 从 OKX 拉取历史并构建数据集
#   MAX_BARS = 'all'  拉到历史尽头（最多，约 2 年 ~100 万根，拉取约 20 分钟）
#   若会话时间紧张，可改成数字，如 '400000'（约 9 个月）。
MAX_BARS = 'all'
!python getdata_btc.py {MAX_BARS} BTC-USDT

In [ ]:
# 3) 训练流水线（预处理 → 训模型1 → 推理 → 预处理 → 训模型2）
#   GASF 按 batch 现算（gaf_transform.py），不预存大数组；两个 train.py 已用时序切分。
import subprocess, sys, os
steps = [
    ('CNN/makedata.py',             '数据预处理(原始窗口)'),
    ('CNN_Transformer/makedata.py', 'CNN_Transformer 标签(DCT)'),
    ('CNN_Transformer/train.py',    '训练【模型1】CNN_Transformer'),
    ('CNN_Transformer/gy.py',       '模型1 推理 → UD 输入'),
    ('UD/makedata.py',              'UD 残差标签'),
    ('UD/train.py',                 '训练【模型2】UD Diffusion'),
]
for script, desc in steps:
    d = os.path.dirname(script) or '.'
    print(f'\n===== {desc} =====', flush=True)
    subprocess.run([sys.executable, os.path.basename(script)], cwd=d, check=True)

In [ ]:
# 4) 收集产物到 /kaggle/working（在 Output 面板下载）
import shutil, os
for src in ['CNN_Transformer/transformer_dct_v1.pth', 'UD/diffusion_delta_v1.pth']:
    if os.path.exists(src):
        shutil.copy(src, '/kaggle/working/' + os.path.basename(src))
        print('✅', src, '->', os.path.getsize(src) // 1024, 'KB')
    else:
        print('❌ 未找到', src)
!ls -la /kaggle/working/*.pth